In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/271 Final Project/fitzpatrick17k.csv')
df.head()

In [ ]:
print(df['label'].unique())
print(df['label'].value_counts())

In [ ]:
#count of images for light and dark skin tone groups
df['fitzpatrick_scale'].value_counts().sort_index()

df['tone_group'] = df['fitzpatrick_scale'].apply(lambda x: 'Dark (IV-VI)' if x >= 4 else 'Light (I-III)')
df['tone_group'].value_counts()

In [ ]:
#underrepresented labels in the darker IV-VI samples
darker_tone = df[df['fitzpatrick_scale'] >= 4].groupby('label').size()
print(darker_tone[darker_tone < 10].sort_values())

In [ ]:
import matplotlib.pyplot as plt

#fitpatrick scale distribution
df['fitzpatrick_scale'].value_counts().sort_index().plot(kind='bar')
plt.title('Images per Fitzpatrick Scale')
plt.show()

#most common labels
df['label'].value_counts().head(20).plot(kind='barh', figsize=(8,8))
plt.title('Top 20 Skin Conditions')
plt.show()

In [ ]:
df_dermaamin = df.dropna(subset=['fitzpatrick_scale', 'label']).copy()
df_dermaamin = df_dermaamin[df_dermaamin['url'].str.contains('dermaamin', na=False)]

print(f"Total DermaAmin rows: {len(df_dermaamin)}")
print(f"Unique labels: {df_dermaamin['label'].nunique()}")
print(f"Tone group split:\n{df_dermaamin['tone_group'].value_counts()}")

In [ ]:
subset_size = 4500
per_group = subset_size // 2

def sample_group(group_df, n_total, seed=42):
    label_counts = group_df['label'].value_counts()
    n_labels = len(label_counts)
    base_per_label = n_total // n_labels

    sampled = []
    for label, count in label_counts.items():
        label_df = group_df[group_df['label'] == label]
        n_sample = min(count, max(1, base_per_label))
        sampled.append(label_df.sample(n=n_sample, random_state=42))

    return pd.concat(sampled)

light_df = df_dermaamin[df_dermaamin['tone_group'] == 'Light (I-III)']
dark_df  = df_dermaamin[df_dermaamin['tone_group'] == 'Dark (IV-VI)']

print(f"Available light: {len(light_df)}, Available dark: {len(dark_df)}")

per_group = min(per_group, len(light_df), len(dark_df))
print(f"Sampling {per_group} per group ({per_group*2} total)")

light_sample = sample_group(light_df, per_group, seed=42)
dark_sample  = sample_group(dark_df,  per_group, seed=42)

subset = pd.concat([light_sample, dark_sample]).drop_duplicates()

print(f"\nFinal subset size: {len(subset)}")
print(f"Light (I-III): {(subset['tone_group'] == 'Light (I-III)').sum()}")
print(f"Dark (IV-VI):  {(subset['tone_group'] == 'Dark (IV-VI)').sum()}")
print(f"Unique labels: {subset['label'].nunique()}")

In [ ]:
import requests
import os
from PIL import Image
from io import BytesIO
import time
from tqdm import tqdm

os.makedirs('/content/drive/MyDrive/271 Final Project/images', exist_ok=True)

URL_COL = 'url'
SAVE_DIR = '/content/drive/MyDrive/271 Final Project/images'

In [ ]:
failed_urls = []

for idx, row in tqdm(subset.iterrows(), total=len(subset)):
    filename = f"{row['md5hash']}.jpg"
    save_path = os.path.join(SAVE_DIR, filename)

    if os.path.exists(save_path):
        continue

    try:
        response = requests.get(row[URL_COL], headers=headers, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img.save(save_path)
        time.sleep(0.1)
    except Exception as e:
        print(f"Failed: {row[URL_COL]} — {e}")
        failed_urls.append(idx)

print(f"\nDownloaded: {len(subset) - len(failed_urls)}")
print(f"Failed: {len(failed_urls)}")

# Add image path to subset
subset['image_path'] = subset['md5hash'].apply(
    lambda h: os.path.join(SAVE_DIR, f"{h}.jpg")
              if os.path.exists(os.path.join(SAVE_DIR, f"{h}.jpg")) else None
)
print(f"Missing images: {subset['image_path'].isna().sum()}")

In [ ]:
# #test subset
# test_subset = subset.head(5)

# for idx, row in test_subset.iterrows():
#     filename = f"{row['md5hash']}.jpg"
#     save_path = os.path.join(SAVE_DIR, filename)

#     try:
#         response = requests.get(row[URL_COL], headers=headers, timeout=10)
#         img = Image.open(BytesIO(response.content)).convert('RGB')
#         img.save(save_path)
#         print(f"✓ Saved: {filename}")
#     except Exception as e:
#         print(f"✗ Failed: {row[URL_COL]} — {e}")

In [ ]:
print(subset[URL_COL].apply(lambda x: 'atlasdermatologico' if 'atlasdermatologico' in str(x) else 'dermaamin').value_counts())

In [ ]:
subset_dermaamin = subset[subset[URL_COL].str.contains('dermaamin', na=False)].copy()
print(f"DermaAmin only subset: {len(subset_dermaamin)}")

In [ ]:
#correlate each image path in drive to respective row
subset['image_path'] = subset['md5hash'].apply(
    lambda h: os.path.join(SAVE_DIR, f"{h}.jpg")
              if os.path.exists(os.path.join(SAVE_DIR, f"{h}.jpg")) else None
)

print(f"Matched: {subset['image_path'].notna().sum()}")
subset[['md5hash', 'label', 'fitzpatrick_scale', 'tone_group', 'image_path']].head()
print(subset['tone_group'].value_counts())

In [ ]:
subset.to_csv('/content/drive/MyDrive/271 Final Project/subset_with_paths.csv', index=False)

In [ ]:
!pip install open_clip_torch

import torch
import open_clip
from PIL import Image
import numpy as np
from tqdm import tqdm

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

embeddings = []
valid_indices = []

for idx, row in tqdm(subset.iterrows(), total=len(subset)):
    try:
        img = Image.open(row['image_path']).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = model.encode_image(img_tensor)
            embedding = embedding / embedding.norm(dim=-1, keepdim=True)  # normalize

        embeddings.append(embedding.cpu().numpy().squeeze())
        valid_indices.append(idx)
    except Exception as e:
        print(f"Failed: {row['image_path']} — {e}")

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
np.save('/content/drive/MyDrive/271 Final Project/embeddings.npy', embeddings)

subset_valid = subset.loc[valid_indices].copy()
subset_valid['embedding_idx'] = range(len(subset_valid))
subset_valid.to_csv('/content/drive/MyDrive/271 Final Project/subset_with_embeddings.csv', index=False)

In [ ]:
print(f"Shape: {embeddings.shape}")
print(f"Sample embedding (first 10 dims): {embeddings[0][:10]}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 6))
for group, color in [('Light (I-III)', 'steelblue'), ('Dark (IV-VI)', 'coral')]:
    mask = subset_valid['tone_group'].values == group
    plt.scatter(reduced[mask, 0], reduced[mask, 1], label=group, alpha=0.5, s=10, c=color)

plt.legend()
plt.title('PCA of OpenCLIP Embeddings by Skin Tone')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
subset_valid.to_csv('/content/drive/MyDrive/271 Final Project/subset_with_paths.csv', index=False)

In [ ]:
#directly load embedddings
import numpy as np
import pandas as pd

embeddings = np.load('/content/drive/MyDrive/271 Final Project/embeddings.npy')
subset_valid = pd.read_csv('/content/drive/MyDrive/271 Final Project/subset_with_paths.csv')

print(f"Embeddings shape: {embeddings.shape}")
print(f"Metadata rows: {len(subset_valid)}")